# 02 - Stage 2 training

The full five-branch model on the real feature store. This is the research result.

Order matters: `00_env_check` must have passed on this image, and `01_baselines` gives the
number this has to beat. Training first and checking later is how you end up with a
result you cannot explain.

In [ ]:
# Unzip the bundle uploaded through the Altair portal, then work from it.
# Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
import glob, json, os, sys, zipfile, pathlib

BUNDLE = next(iter(sorted(glob.glob("*.zip")) + sorted(glob.glob("/data/**/*.zip", recursive=True))), None)
if BUNDLE and not pathlib.Path("src/mapguard").exists():
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(".")
    print("unpacked", BUNDLE)

sys.path.insert(0, "src")
TASK = [p for p in glob.glob("*/BUNDLE_MANIFEST.json")]
if TASK:
    manifest = json.load(open(TASK[0]))
    print(json.dumps({k: manifest[k] for k in ("task","git_sha","git_dirty","generated")}, indent=2))
    if manifest.get("git_dirty"):
        print("WARNING: bundle was built from a dirty tree; git_sha does not fully describe it")

In [ ]:
import json, glob, os, datetime
import numpy as np
import pyarrow.parquet as pq
import tensorflow as tf
from tensorflow import keras

print("tf", tf.__version__, "| keras", keras.__version__, "| GPU", tf.config.list_physical_devices("GPU"))

changesets = pq.read_table("data/interim/changesets.parquet").to_pandas()
edits      = pq.read_table("data/interim/edits.parquet").to_pandas()
user_seq   = pq.read_table("data/interim/user_seq.parquet").to_pandas()
print(len(changesets), "changesets |", len(edits), "sampled edits |", len(user_seq), "history steps")

## Assemble the padded arrays

`build_batch` is where masking bugs are born - each changeset holds a different number of
edits, and if the mask does not line up with the padding the model attends to zeros. The
alignment is unit-tested locally (`tests/test_dataset.py`); the assertion below is the
belt to that braces.

In [ ]:
from mapguard.training.dataset import build_batch, standardise, EDIT_FEATURES, TABULAR_FEATURES

batch = build_batch(changesets, edits, user_seq, max_edits=256, max_user_steps=200)
print("edit_features", batch.edit_features.shape, "| edit_mask", batch.edit_mask.shape)
print("mean real edits per changeset:", batch.edit_mask.sum(axis=1).mean().round(2))
assert np.all(batch.edit_features[batch.edit_mask == 0] == 0), "padded slots carry signal"

In [ ]:
from mapguard.evaluation.splits import user_disjoint_split, random_split, assert_user_disjoint

y = batch.y
r1b = user_disjoint_split(list(batch.uids), y)
assert_user_disjoint(r1b, list(batch.uids))
print("R1b:", r1b.summary(y))

tr, va, te = r1b.train, r1b.val, r1b.test
tab_tr, tab_va, tab_te = standardise(batch.tabular[tr], batch.tabular[va], batch.tabular[te])

## Build, verify masking on the REAL model, then train

In [ ]:
from mapguard.models.stage2 import build_model, focal_loss, verify_masking

CFG = dict(max_edits=batch.edit_features.shape[1],
           edit_features=batch.edit_features.shape[2],
           text_len=batch.text.shape[1], vocab_size=int(batch.text.max()) + 2,
           user_steps=batch.user_sequence.shape[1], user_features=batch.user_sequence.shape[2],
           tabular_features=batch.tabular.shape[1])

model = build_model(branches=("B2","B3","B4","B5"), raster=None, **CFG)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss={"p_damaging": focal_loss(), "edit_attention_weight": None},
              metrics={"p_damaging": [keras.metrics.AUC(curve="PR", name="pr_auc")]})
model.summary()

order = [t.name.split(":")[0] for t in model.inputs]
def pack(mask, tab):
    d = dict(batch.inputs()); d["tabular"] = tab
    return [np.asarray(d[k])[mask] if k != "tabular" else tab for k in order]

check = verify_masking(model, {k: v[:64] for k, v in dict(batch.inputs()).items()})
print("masking on the real model:", check)
assert check["masking_holds"]

In [ ]:
os.makedirs("experiments/checkpoints", exist_ok=True)
cb = [
    keras.callbacks.ModelCheckpoint("experiments/checkpoints/stage2_epoch{epoch:02d}.weights.h5",
                                    save_weights_only=True),
    keras.callbacks.EarlyStopping(monitor="val_p_damaging_pr_auc", mode="max",
                                  patience=5, restore_best_weights=True),
]
# Early stopping on validation PR-AUC, never accuracy. The classes are balanced here but
# the operating point is not, and accuracy would happily pick the wrong model.

hist = model.fit(
    pack(tr, tab_tr), {"p_damaging": y[tr]},
    validation_data=(pack(va, tab_va), {"p_damaging": y[va]}),
    epochs=40, batch_size=64, callbacks=cb, verbose=2,
)

In [ ]:
from mapguard.evaluation.metrics import summary as metric_summary, bootstrap_ci

p, w = model.predict(pack(te, tab_te), verbose=0)
p = p.ravel()
res = metric_summary(y[te], p)
res["pr_auc_ci95"] = bootstrap_ci(y[te], p)
print(json.dumps(res, indent=2, default=float))

## Save everything a result needs to be reproducible

In [ ]:
manifest = json.load(open(glob.glob("*/BUNDLE_MANIFEST.json")[0]))
model.save("experiments/checkpoints/stage2_final.keras")
out = {
    "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "git_sha": manifest["git_sha"],
    "feature_schema": manifest.get("feature_schema"),
    "branches": list(model.branch_config),
    "config": CFG,
    "split": r1b.summary(y),
    "seed": 20260904,
    "checkpoint": "experiments/checkpoints/stage2_final.keras",
    "masking_check": check,
    "epochs_run": len(hist.history["loss"]),
    "test": res,
}
os.makedirs("results", exist_ok=True)
json.dump(out, open("results/stage2_r1b.json", "w"), indent=2, default=float)
print(json.dumps({k: v for k, v in out.items() if k != "test"}, indent=2, default=float))
print("\nDownload results/stage2_r1b.json and experiments/checkpoints/stage2_final.keras")